# 09 — Over-/under-fitting and the generalization gap

*Module 00 · Getting Started — notebook 9 of 11*

The central tension of machine learning: a model can be too simple to capture the pattern, or so complex it memorizes noise. This notebook makes that tension visible, names it, and gives you the curves to diagnose it.

**Prerequisites:** 04 (the train/test split), 06 (accuracy).

**What you'll be able to do**
- Recognize **underfitting** and **overfitting**, and the **generalization gap** between train and test.
- Explain the **bias-variance** trade-off behind that gap.
- Read a **complexity curve** and a **learning curve** to diagnose a model.
- Say when more data helps — and when it will not.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import make_moons
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

from ml_course import colors, viz

np.random.seed(0)
viz.use_course_style()


def flexible_boundary(degree):
    '''A classifier whose complexity is set by the polynomial degree.

    Plumbing only: it expands the two features to polynomials of the given degree, puts them on a
    common scale, and fits a plain linear classifier (logistic regression — chapter 03; scaling —
    notebook 11). The one knob we turn in this notebook is the degree.
    '''
    return make_pipeline(
        PolynomialFeatures(degree),
        StandardScaler(),
        LogisticRegression(C=1e6, max_iter=5000),
    )

## Why a new dataset

Our penguins separate so cleanly that any reasonable model gets them all right — there is no room to *see* overfitting. To watch a model overfit, we need a harder problem. So this one notebook steps away from penguins and uses a small synthetic dataset of two interleaving "moons" with deliberate noise. (We return to penguins in notebook 11.)

In [ ]:
X, y = make_moons(n_samples=300, noise=0.30, random_state=0)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=0, stratify=y
)

fig, ax = plt.subplots(figsize=(7, 5.5))
for cls, color in zip([0, 1], colors.CLASS_CYCLE, strict=False):
    pts = X_train[y_train == cls]
    ax.scatter(
        pts[:, 0], pts[:, 1], color=color, edgecolor=colors.COLORS['text'],
        linewidth=0.4, s=35, label=f'class {cls}',
    )
ax.set_xlabel('x1')
ax.set_ylabel('x2')
ax.set_title('A harder problem: two interleaving moons, with noise')
ax.legend()
plt.show()

### Read the figure

Two classes wound around each other, blurred by noise. No straight line cuts them cleanly — a useful model has to *bend*. That makes this a good place to ask: how much bending is right? Too little and we miss the curve; too much and we trace every noisy wiggle. We need a knob for "how complex", and a way to tell whether we have turned it too far.

## A dial for complexity

We will turn one knob: the **polynomial degree**. Degree 1 lets the model draw only a straight boundary; higher degrees let it curve more and more. The engine that actually fits the boundary expands the two features to polynomials, puts them on a common scale, and runs a plain linear classifier — that is logistic regression (chapter 03) with standardisation (notebook 11), and you can treat it as a black box here. The only thing we study in this notebook is the effect of the **degree**.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, degree in zip(axes, [1, 3, 9], strict=False):
    model = flexible_boundary(degree).fit(X_train, y_train)
    viz.plot_decision_boundary(model, X_train, y_train, resolution=200, ax=ax)
    ax.set_title(f'degree {degree}')
plt.show()

### Read the figure

At **degree 1** the boundary is a straight line — too rigid for these moons, so it misses whole arcs of points: this is **underfitting**. At **degree 9** the boundary writhes around individual points, carving out little islands to capture noise: this is **overfitting**. **Degree 3** bends enough to follow the moons without chasing the noise. The same data, three very different rules — set only by the degree.

In [ ]:
degrees = list(range(1, 10))
train_err, test_err = [], []
for degree in degrees:
    model = flexible_boundary(degree).fit(X_train, y_train)
    train_err.append(1 - model.score(X_train, y_train))
    test_err.append(1 - model.score(X_test, y_test))

viz.plot_train_test_curve(
    degrees, train_err, test_err, xlabel='polynomial degree', ylabel='error rate'
)
plt.show()
print(f'lowest test error at degree {degrees[int(np.argmin(test_err))]}')

### Read the figure

Watch the two curves. The **training** error trends downward as the degree rises — a more flexible model can fit its own training data ever more closely, here heading toward memorizing it (with a small wobble at the start: the degree-1 and degree-2 fits are about as good as each other on the training set). But the **test** error traces a **U**: high when we underfit (degree 1–2), lowest around degree 3 on this split, then climbing again as we overfit. The vertical distance between the curves is the **generalization gap** — how much worse the model does on new data than on the data it trained on. It is small at the sweet spot and grows as we overfit. (Two honest caveats: the gap is a symptom, *related to* variance but not the same quantity — the next section says why; and the exact best degree wobbles with the random split, which is what notebook 10 sets out to handle.)

## Why the U? Bias and variance

The U comes from two competing ways of being wrong.

- A model that is **too simple** cannot represent the true pattern no matter the data — it has high **bias**, and it underfits (both training and test error stay high).
- A model that is **too complex** bends to the particular noise in *this* training set — it has high **variance**, and it overfits (training error low, test error high).

Increasing complexity lowers bias but raises variance; the best degree trades them off. Formally, the expected test error splits into **bias² + variance + irreducible noise**, where bias and variance are averages over many possible training sets — which is why this is the explanation *behind* the single-split gap we measured, not a relabelling of it.

## A second diagnostic: the learning curve

The complexity curve fixes the data and varies the model. The **learning curve** does the opposite: it fixes the model (here, our degree-6 one) and varies the *amount of training data*, plotting training and test error as the training set grows. We grow the training set in steps and, each time, score on the same sealed test set — no new machinery, only the split from notebook 04.

In [ ]:
rng = np.random.default_rng(0)
sizes = [30, 60, 100, 140, len(X_train)]
lc_train, lc_test = [], []
for n in sizes:
    idx = rng.choice(len(X_train), size=n, replace=False)
    model = flexible_boundary(6).fit(X_train[idx], y_train[idx])
    lc_train.append(1 - model.score(X_train[idx], y_train[idx]))
    lc_test.append(1 - model.score(X_test, y_test))

viz.plot_train_test_curve(sizes, lc_train, lc_test, xlabel='training examples', ylabel='error rate')
plt.show()

### Read the figure

With very few examples the model fits them all — a degree-6 boundary can thread through a handful of points, so training error sits near zero — yet it generalizes poorly, so the gap is wide. As the training set grows, training error rises (it can no longer memorize every point) and the test error trends down toward it: the curves move together and the gap narrows. (With small samples the test curve is noisy and may tick up before settling, so read the trend, not each step.) The lesson: for a model that is overfitting (high variance), **more data helps**. For one that is underfitting (high bias), the curves would meet at a high error and more data would not rescue it — you would reach for a more flexible model instead.

## Going further (optional)

We said the generalization gap is related to variance without being the same number. Here is the picture behind "variance": it is how much the fitted model *moves around* when the training set changes. We can see it directly — refit the flexible (degree-9) model on several random resamples of the data and overlay the boundaries. This section is optional; skip it without losing the thread.

In [ ]:
xx, yy = np.meshgrid(
    np.linspace(X_train[:, 0].min() - 0.5, X_train[:, 0].max() + 0.5, 200),
    np.linspace(X_train[:, 1].min() - 0.5, X_train[:, 1].max() + 0.5, 200),
)
grid = np.c_[xx.ravel(), yy.ravel()]

fig, ax = plt.subplots(figsize=(7, 6))
for cls, color in zip([0, 1], colors.CLASS_CYCLE, strict=False):
    pts = X_train[y_train == cls]
    ax.scatter(pts[:, 0], pts[:, 1], color=color, s=12, alpha=0.3, edgecolor='none')

resample_rng = np.random.default_rng(1)
for _ in range(6):
    idx = resample_rng.choice(len(X_train), size=len(X_train), replace=True)
    model = flexible_boundary(9).fit(X_train[idx], y_train[idx])
    zz = model.predict(grid).reshape(xx.shape)
    ax.contour(
        xx, yy, zz, levels=[0.5], colors=[colors.COLORS['highlight']], linewidths=1.2, alpha=0.7
    )

ax.set_xlabel('x1')
ax.set_ylabel('x2')
ax.set_title('Degree-9 boundary across 6 resamples — the spread is variance')
plt.show()

### Read the figure

Each line is the degree-9 boundary fitted on a different resample of the same data. They scatter wildly — a small change in the training set throws the boundary around. *That* spread is **variance**, and it is why a too-complex model generalizes poorly: it is fitting the sample, not the signal. A degree-1 boundary refit the same way would barely move (low variance) but would sit in the wrong place for these moons (high bias).

## Your turn

1. On a complexity curve, training error falls steadily while test error bottoms out at degree 4 and then rises. Which degrees are underfitting, which are overfitting, and which would you pick?
2. A model scores 100% on training data and 70% on test data. What is happening, and name one thing you could try to close the gap.
3. A model underfits — both training and test error are high and close together. Would collecting more data help? Why or why not?

Notebook 10 closes the one loose end below.

## What you built

You met the tension that shapes every model in this course:

- You can tell **underfitting** from **overfitting**, and read the **generalization gap** between training and test error.
- You can explain the **bias-variance** trade-off that produces the U-shaped test error.
- You can read a **complexity curve** and a **learning curve**, and decide whether more data, or more (or less) flexibility, is the fix.

**Vocabulary added:** underfitting, overfitting, model complexity, generalization gap, bias, variance, learning curve.

One honest loose end: we picked the best degree by looking at the *test* error. Do that repeatedly and the test set quietly stops being a fair judge. Notebook 10 fixes exactly this, with cross-validation.

## References

- G. James, D. Witten, T. Hastie, R. Tibshirani, *An Introduction to Statistical Learning*, 2nd ed., Springer, 2021 — §2.2 (assessing model accuracy; the bias-variance trade-off). DOI: 10.1007/978-1-0716-1418-1
- P. Domingos (2012), *A Few Useful Things to Know about Machine Learning*, Communications of the ACM 55(10):78–87. DOI: 10.1145/2347736.2347755
- A. Géron, *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow*, 3rd ed., O'Reilly, 2022 — chapter 4 (learning curves; the bias-variance trade-off).

---
Previous: **08 — Scores, thresholds, ROC & AUC** · Next: **10 — Validating honestly: cross-validation**